In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

public interface ICalculable
{
    void CalculateTotal();
}

public interface ITrackable
{
    void MarkAsPaid();
}

public interface IPayable
{
    void Pay(decimal amount);
}

public class LineItem
{
    public string Description { get; set; }
    public decimal Quantity { get; set; }
    public decimal UnitPrice { get; set; }
    public bool IsReturned { get; set; }

    public decimal Discount { get; set; }

    public decimal Total => Quantity * UnitPrice - Discount;

    public LineItem(string desc, decimal qty, decimal price)
    {
        Description = desc;
        Quantity = qty;
        UnitPrice = price;
    }

    public decimal GetTotal(bool includeTax)
    {
        var total = Total;
        return includeTax ? total * 1.2m : total;
    }
}

public class Invoice : ICalculable, ITrackable, IPayable
{
    public string InvoiceNumber { get; protected set; }
    public DateTime IssueDate { get; protected set; }
    public decimal TotalAmount { get; protected set; }

    public string Currency { get; set; } = "EUR";
    public decimal Discount { get; set; }
    public bool IsPaid { get; private set; }
    public string CustomerName { get; set; }

    protected List<LineItem> LineItems { get; } = new();
    
    // Событие для отслеживания оплаты
    public event Action<Invoice> InvoicePaid;

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
    }

    void ICalculable.CalculateTotal()
    {
        TotalAmount = LineItems.Sum(x => x.Total) - Discount;
    }

    void ITrackable.MarkAsPaid()
    {
        IsPaid = true;
        // Генерация события, когда инвойс оплачен
        InvoicePaid?.Invoke(this);
    }

    void IPayable.Pay(decimal amount)
    {
        if (amount >= TotalAmount)
        {
            IsPaid = true;
            Console.WriteLine($"Invoice {InvoiceNumber} paid.");
            // Генерация события, когда инвойс оплачен
            InvoicePaid?.Invoke(this);
        }
        else
        {
            Console.WriteLine($"Insufficient payment for Invoice {InvoiceNumber}");
        }
    }

    public void ApplyDiscount(decimal value)
    {
        Discount = value;
        CalculateTotal();
    }

    public void PrintInfo()
    {
        Console.WriteLine($"{InvoiceNumber} | {TotalAmount:C} | Paid: {IsPaid}");
    }

    public IReadOnlyList<LineItem> GetLineItems() => LineItems.AsReadOnly();
}

public class GoodsInvoice : Invoice, ICalculable, ITrackable, IPayable
{
    public DateTime SupplyDate { get; set; }
    public decimal DeliveryCost { get; set; }

    public GoodsInvoice(string number, DateTime issueDate, DateTime supplyDate)
        : base(number, issueDate)
    {
        SupplyDate = supplyDate;
    }

    void ICalculable.CalculateTotal()
    {
        base.CalculateTotal();
        TotalAmount += DeliveryCost;
        Console.WriteLine("[GoodsInvoice] Delivery included");
    }

    void ITrackable.MarkAsPaid()
    {
        IsPaid = true;
        Console.WriteLine("[GoodsInvoice] Marked as paid");
    }

    void IPayable.Pay(decimal amount)
    {
        if (amount >= TotalAmount)
        {
            IsPaid = true;
            Console.WriteLine($"GoodsInvoice {InvoiceNumber} paid.");
        }
        else
        {
            Console.WriteLine($"Insufficient payment for GoodsInvoice {InvoiceNumber}");
        }
    }

    public void AddLine(LineItem item, bool fragile)
    {
        if (fragile)
            item.UnitPrice += 5;

        base.AddLine(item);
    }
}

public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }
    public decimal HourlyRateBonus { get; set; }

    public ServiceInvoice(string number, DateTime issueDate, DateTime serviceDate)
        : base(number, issueDate)
    {
        ServiceDate = serviceDate;
    }

    void ICalculable.CalculateTotal()
    {
        TotalAmount = LineItems.Sum(x => x.Total + HourlyRateBonus);
        TotalAmount -= Discount;
        Console.WriteLine("[ServiceInvoice] Bonus applied");
    }

    void ITrackable.MarkAsPaid()
    {
        IsPaid = true;
        Console.WriteLine("[ServiceInvoice] Marked as paid");
    }

    void IPayable.Pay(decimal amount)
    {
        if (amount >= TotalAmount)
        {
            IsPaid = true;
            Console.WriteLine($"ServiceInvoice {InvoiceNumber} paid.");
        }
        else
        {
            Console.WriteLine($"Insufficient payment for ServiceInvoice {InvoiceNumber}");
        }
    }
}

public class Repository<T>
{
    private readonly List<T> _items = new();

    public void Add(T item) => _items.Add(item);
    public IEnumerable<T> GetAll() => _items;
    public T Find(Func<T, bool> predicate) => _items.FirstOrDefault(predicate);
}

public delegate void InvoiceProcessedDelegate(Invoice invoice);
public class InvoiceProcessor
{
    public event InvoiceProcessedDelegate InvoiceProcessed;

    public void ProcessInvoice(Invoice invoice)
    {
        invoice.CalculateTotal();
        invoice.PrintInfo();
        InvoiceProcessed?.Invoke(invoice); // Генерация события после обработки
    }
}

class Program
{
    static void Main()
    {
        var goodsInvoice = new GoodsInvoice("G1", DateTime.Now, DateTime.Now)
        {
            DeliveryCost = 50
        };
        
        goodsInvoice.InvoicePaid += InvoicePaidHandler; // Подписка на событие

        // Создание инвойса и добавление строк
        var lineItem1 = new LineItem("Laptop", 1, 1000);
        goodsInvoice.AddLine(lineItem1);

        // Создание процессора инвойсов
        var processor = new InvoiceProcessor();
        processor.InvoiceProcessed += Processor_InvoiceProcessed; // Подписка на событие
        processor.ProcessInvoice(goodsInvoice);

        // Оплата инвойса
        goodsInvoice.Pay(1050);

        // Создание репозитория и добавление инвойсов
        var invoiceRepo = new Repository<Invoice>();
        invoiceRepo.Add(goodsInvoice);

        // Поиск инвойса по номеру
        var foundInvoice = invoiceRepo.Find(i => i.InvoiceNumber == "G1");
        Console.WriteLine($"Found Invoice: {foundInvoice?.InvoiceNumber}");
    }

    // Обработчик события "InvoicePaid"
    private static void InvoicePaidHandler(Invoice invoice)
    {
        Console.WriteLine($"Invoice {invoice.InvoiceNumber} has been paid!");
    }

    // Обработчик события "InvoiceProcessed"
    private static void Processor_InvoiceProcessed(Invoice invoice)
    {
        Console.WriteLine($"Invoice {invoice.InvoiceNumber} has been processed.");
    }
}